In [1]:
import numpy as np
import xarray as xr
from typing import Union, Optional
import warnings

#### Constant

In [2]:
EULER = np.exp(1)  # ≈ 2.718281828
RWI_EXPONENT = 1 / EULER  # ≈ 0.367879441

#### n-value from ROI

In [3]:
def calculate_n_from_roi(
    green_band: xr.DataArray,
    roi_mask: Optional[xr.DataArray] = None,
    nodata: Optional[float] = None,
    min_valid_pixels: int = 100,
    filter_extremes: bool = True,  # NEW: filter outliers for n calculation
    percentile_range: tuple = (1, 99)
) -> float:
    """
    Calculate RWI normalization factor 'n' using ROI.

    Note: For n calculation, extreme values (> 99th percentile) can be filtered
    to avoid bias from anomalies, but input values are NOT masked.
    """
    green_arr = green_band.values.copy()

    if roi_mask is not None:
        green_arr = np.where(roi_mask.values, green_arr, np.nan)

    if nodata is not None:
        green_arr = np.where(green_arr == nodata, np.nan, green_arr)

    # Basic validity: finite and non-negative (accept values > 1)
    valid_mask = np.isfinite(green_arr) & (green_arr >= 0)

    # Optional: filter extreme outliers for robust n calculation
    if filter_extremes:
        valid_values = green_arr[valid_mask]
        if len(valid_values) > 100:
            p_low, p_high = np.percentile(valid_values, percentile_range)
            valid_mask = valid_mask & (green_arr >= p_low) & (green_arr <= p_high)

    green_valid = green_arr[valid_mask]

    if len(green_valid) < min_valid_pixels:
        # Fallback: use all non-negative finite values
        green_valid = green_arr[np.isfinite(green_arr) & (green_arr >= 0)]
        if len(green_valid) < min_valid_pixels:
            raise ValueError(f"Insufficient valid pixels: {len(green_valid)} < {min_valid_pixels}")

    median_g = np.median(green_valid)
    median_g_exp = np.median(np.power(green_valid, RWI_EXPONENT))

    if median_g == 0:
        return 1.0

    return float(median_g_exp / median_g)


def calculate_n_from_image(
    green_band: xr.DataArray,
    nodata: Optional[float] = None,
    min_valid_fraction: float = 0.01,
    filter_extremes: bool = True,
    percentile_range: tuple = (1, 99)
) -> float:
    """Calculate n from entire image with optional outlier filtering."""
    green_arr = green_band.values.ravel().copy()

    if nodata is not None:
        green_arr = np.where(green_arr == nodata, np.nan, green_arr)

    valid_mask = np.isfinite(green_arr) & (green_arr >= 0)

    if filter_extremes:
        valid_values = green_arr[valid_mask]
        if len(valid_values) > 1000:
            p_low, p_high = np.percentile(valid_values, percentile_range)
            valid_mask = valid_mask & (green_arr >= p_low) & (green_arr <= p_high)

    green_valid = green_arr[valid_mask]

    if len(green_valid) / len(green_arr) < min_valid_fraction:
        green_valid = green_arr[np.isfinite(green_arr) & (green_arr >= 0)]

    if len(green_valid) == 0:
        return 1.0

    median_g = np.median(green_valid)
    median_g_exp = np.median(np.power(green_valid, RWI_EXPONENT))

    if median_g == 0:
        return 1.0

    return float(median_g_exp / median_g)

#### n-value from sample points

In [4]:
def calculate_n_from_samples(
    green_values: Union[np.ndarray, list],
    sample_mask: Optional[Union[np.ndarray, list]] = None,
    nodata: Optional[Union[int, float]] = None,
    min_valid_samples: int = 50,
    filter_extremes: bool = True,  # NEW: filter outliers for robust n calculation
    percentile_range: tuple = (1, 99)  # NEW: percentile range for filtering
) -> float:
    """
    Calculate the RWI normalization factor 'n' using sample point values.

    Parameters
    ----------
    green_values : array-like
        1D array of Green band reflectance values.
        Values >= 0 are accepted (not limited to [0-1]).
        High values (> 1) may indicate solar panels or other anomalies.
    sample_mask : array-like, optional
        Boolean array indicating which samples to include (True = valid).
        If None, all samples are used.
    nodata : int or float, optional
        Value representing no-data samples. If None, uses NaN masking.
    min_valid_samples : int, default=50
        Minimum number of valid samples required for reliable median calculation.
    filter_extremes : bool, default=True
        If True, filters extreme outliers (outside percentile_range) when
        calculating n to avoid bias from anomalies. Input values are NOT masked.
    percentile_range : tuple, default=(1, 99)
        Percentile range [low, high] for filtering extreme values when
        filter_extremes=True. Only affects n calculation, not output values.

    Returns
    -------
    float
        The normalization factor n = median(Green^(1/e)) / median(Green)

    Raises
    ------
    ValueError
        If insufficient valid samples are provided.

    Notes
    -----
    - Samples MUST include both water AND non-water observations for proper calibration.
    - This method is useful when working with field validation points or
      stratified random samples.
    - Values > 1 in input are accepted but can be filtered for n calculation
      via filter_extremes parameter to avoid bias from anomalies.
    """
    # Convert to numpy array
    green_arr = np.asarray(green_values, dtype=float).copy()

    # Apply sample mask if provided
    if sample_mask is not None:
        mask = np.asarray(sample_mask, dtype=bool)
        green_arr = green_arr[mask]

    # Handle nodata if specified
    if nodata is not None:
        green_arr = np.where(green_arr == nodata, np.nan, green_arr)

    # CORRECTION: Remove invalid values (negative or non-finite)
    # ACCEPT values > 1 (solar panels, high-reflectance anomalies)
    valid_mask = (green_arr >= 0) & np.isfinite(green_arr)

    # Optional: filter extreme outliers for robust n calculation
    if filter_extremes:
        valid_values = green_arr[valid_mask]
        if len(valid_values) > min_valid_samples * 2:  # Only filter if enough samples
            p_low, p_high = np.percentile(valid_values, percentile_range)
            valid_mask = valid_mask & (green_arr >= p_low) & (green_arr <= p_high)

    green_valid = green_arr[valid_mask]

    # Check minimum sample count
    if len(green_valid) < min_valid_samples:
        # Fallback: try without extreme filtering
        if filter_extremes:
            green_valid = green_arr[
                (green_arr >= 0) & np.isfinite(green_arr)
            ]

        if len(green_valid) < min_valid_samples:
            raise ValueError(
                f"Insufficient valid samples: {len(green_valid)} < {min_valid_samples}. "
                "Ensure samples include both water and non-water observations."
            )

    # Calculate medians
    median_green = np.median(green_valid)
    median_green_exp = np.median(np.power(green_valid, RWI_EXPONENT))

    # Calculate and return n
    if median_green == 0:
        return 1.0

    n_factor = median_green_exp / median_green

    return float(n_factor)

#### n-value from scene

In [5]:
def calculate_n_from_image(
    green_band: xr.DataArray,
    nodata: Optional[Union[int, float]] = None,
    min_valid_fraction: float = 0.01,
    filter_extremes: bool = True,  # NEW: filter outliers for robust n calculation
    percentile_range: tuple = (1, 99),  # NEW: percentile range for filtering
    max_pixels_for_exact: int = 1000000  # RENAMED: was chunk_size
) -> float:
    """
    Calculate the RWI normalization factor 'n' using all valid pixels in the image.

    Parameters
    ----------
    green_band : xr.DataArray
        Green band reflectance values.
        Values >= 0 are accepted (not limited to [0-1]).
        High values (> 1) may indicate solar panels or other anomalies.
    nodata : int or float, optional
        Value representing no-data pixels. If None, uses NaN masking.
    min_valid_fraction : float, default=0.01
        Minimum fraction of valid pixels required (relative to total image size).
    filter_extremes : bool, default=True
        If True, filters extreme outliers (outside percentile_range) when
        calculating n to avoid bias from anomalies. Input values are NOT masked.
    percentile_range : tuple, default=(1, 99)
        Percentile range [low, high] for filtering extreme values when
        filter_extremes=True. Only affects n calculation, not output values.
    max_pixels_for_exact : int, default=1000000
        Maximum number of pixels for exact median calculation.
        If len(green_valid) > this value, uses random sampling for efficiency.

    Returns
    -------
    float
        The normalization factor n = median(Green^(1/e)) / median(Green)

    Raises
    ------
    ValueError
        If insufficient valid pixels are found in the image.

    Notes
    -----
    - This method is suitable for homogeneous scenes or when no ROI is defined.
    - For heterogeneous scenes (e.g., urban + water), prefer ROI-based calculation.
    - Clouds and shadows should be masked BEFORE calling this function.
    - Values > 1 in input are accepted but can be filtered for n calculation
      via filter_extremes parameter to avoid bias from anomalies.
    - Large images use random sampling for efficient median calculation.
    """
    # Ensure green_band is 2D (y, x)
    if green_band.ndim == 3 and green_band.sizes.get('band') == 1:
        green_band = green_band.squeeze(dim='band', drop=True)

    # Flatten to 1D for efficient processing
    green_flat = green_band.values.ravel().copy()  # .copy() to avoid side effects

    # Handle nodata if specified
    if nodata is not None:
        green_flat = np.where(green_flat == nodata, np.nan, green_flat)

    # CORRECTION: Remove invalid values (negative or non-finite)
    # ACCEPT values > 1 (solar panels, high-reflectance anomalies)
    valid_mask = (green_flat >= 0) & np.isfinite(green_flat)

    # Optional: filter extreme outliers for robust n calculation
    if filter_extremes:
        valid_values = green_flat[valid_mask]
        if len(valid_values) > 1000:  # Only filter if enough values
            p_low, p_high = np.percentile(valid_values, percentile_range)
            valid_mask = valid_mask & (green_flat >= p_low) & (green_flat <= p_high)

    green_valid = green_flat[valid_mask]

    # Check minimum valid fraction
    total_pixels = green_band.size
    valid_fraction = len(green_valid) / total_pixels

    if valid_fraction < min_valid_fraction:
        # Fallback: try without extreme filtering
        if filter_extremes:
            green_valid = green_flat[
                (green_flat >= 0) & np.isfinite(green_flat)
            ]
            valid_fraction = len(green_valid) / total_pixels

        if valid_fraction < min_valid_fraction:
            raise ValueError(
                f"Insufficient valid pixels: {valid_fraction*100:.2f}% < {min_valid_fraction*100:.2f}%. "
                "Ensure clouds and invalid values are properly masked."
            )

    # For very large arrays, use random sampling for efficient median calculation
    if len(green_valid) > max_pixels_for_exact:
        # Approximate median using random sampling (statistically robust for large N)
        np.random.seed(42)  # For reproducibility
        sample_idx = np.random.choice(
            len(green_valid),
            size=max_pixels_for_exact,
            replace=False
        )
        green_sample = green_valid[sample_idx]
        median_green = np.median(green_sample)
        median_green_exp = np.median(np.power(green_sample, RWI_EXPONENT))
    else:
        # Exact median for smaller arrays
        median_green = np.median(green_valid)
        median_green_exp = np.median(np.power(green_valid, RWI_EXPONENT))

    # Calculate and return n
    if median_green == 0:
        return 1.0

    n_factor = median_green_exp / median_green

    return float(n_factor)

#### calculate RWI

In [6]:
def calculate_rwi(
    green: xr.DataArray,
    swir1: xr.DataArray,
    n_factor: Optional[float] = None,
    roi: Optional[xr.DataArray] = None,
    method: str = 'roi',  # 'roi', 'samples', or 'image'
    preserve_anomalies: bool = True  # NEW: keep values > 1
) -> xr.DataArray:
    """
    Calculate the Rescaled Water Index (RWI).

    Parameters
    ----------
    green : xr.DataArray
        Green band reflectance. Values >= 0 accepted (not limited to [0-1]).
        High values (> 1) may indicate solar panels or other anomalies.
    swir1 : xr.DataArray
        SWIR1 band reflectance. Values >= 0 accepted (not limited to [0-1]).
    n_factor : float, optional
        Pre-computed normalization factor. If None, calculates automatically.
    roi : xr.DataArray, optional
        ROI mask for automatic n calculation (required if method='roi').
    method : str, default='roi'
        Method for calculating n: 'roi', 'samples', or 'image'.
    preserve_anomalies : bool, default=True
        If True, preserves output values outside [-1, +1] range (e.g., solar panels).
        If False, clips output to [-1, +1] for standard water mapping.

    Returns
    -------
    xr.DataArray
        RWI values. Typical range is [-1, +1], but values may exceed this range
        when input bands contain high-reflectance anomalies (e.g., solar panels).
        Positive values indicate water probability; values > 1 may indicate anomalies.
    """
    # Calculate n if not provided
    if n_factor is None:
        if method == 'roi':
            if roi is None:
                raise ValueError("ROI mask required when method='roi' and n_factor is None")
            n_factor = calculate_n_from_roi(green, roi)
        elif method == 'samples':
            raise NotImplementedError(
                "Sample-based n calculation requires point extraction. "
                "Please compute n_factor externally and pass as argument."
            )
        elif method == 'image':
            n_factor = calculate_n_from_image(green)
        else:
            raise ValueError(f"Unknown method: {method}. Use 'roi', 'samples', or 'image'.")

    # Apply RWI formula
    # Note: np.power and arithmetic operations preserve all finite values
    green_exp = np.power(green, RWI_EXPONENT)
    green_adjusted = green_exp / n_factor

    with np.errstate(divide='ignore', invalid='ignore'):
        rwi = (green_adjusted - swir1) / (green_adjusted + swir1)

    # CORRECTION: Only mask truly invalid results (inf, nan), NOT anomalous values
    rwi = xr.where(np.isfinite(rwi), rwi, np.nan)

    # Optional: Clip to [-1, +1] if user wants standard water mapping behavior
    if not preserve_anomalies:
        rwi = rwi.clip(-1, 1)

    # Add metadata
    rwi.attrs['long_name'] = 'Rescaled Water Index'
    rwi.attrs['formula'] = '(Green^(1/e)/n - SWIR1) / (Green^(1/e)/n + SWIR1)'
    rwi.attrs['n_factor'] = float(n_factor)  # Ensure JSON-serializable
    rwi.attrs['exponent'] = float(RWI_EXPONENT)
    rwi.attrs['reference'] = 'Justiniano et al. (2025), IEEE J-STARS'
    rwi.attrs['doi'] = 'https://doi.org/10.1109/JSTARS.2025.3562089'
    rwi.attrs['preserve_anomalies'] = preserve_anomalies
    if preserve_anomalies:
        rwi.attrs['note'] = 'Values > 1 may indicate solar panels or high-reflectance anomalies'

    return rwi